In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
PROC = ROOT / 'datasets' / 'processed'
OUT = PROC / 'module1_bus'
OUT.mkdir(parents=True, exist_ok=True)

bus = pd.read_parquet(OUT / 'bus_gps_points.parquet')
print('bus rows', len(bus))
print(bus.columns.tolist())
print(bus.head(3))

Mounted at /content/drive
bus rows 924375
['route_code', 'month', 'day', 'node', 'bus_number', 'direction', 'time_raw', 'source_file', 'stop_sequence', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'node_as_sequence', 'node_as_rank']
   route_code    month   day  node  bus_number direction      time_raw  \
0         425  Month_1  Day1    16  DL1PC 5377        up  05:24:26:000   
1         425  Month_1  Day1    17  DL1PC 5377        up  05:24:56:000   
2         425  Month_1  Day1    18  DL1PC 5377        up  05:26:36:000   

             source_file  stop_sequence  stop_id                    stop_name  \
0  Month_1/425/Day1.xlsx           16.0   1521.0  Govindpuri Extn Gali No. 16   
1  Month_1/425/Day1.xlsx           17.0   1522.0            DDA Flats Kalkaji   
2  Month_1/425/Day1.xlsx           18.0   1524.0            DDA Flats KalkaJi   

    stop_lat   stop_lon  node_as_sequence  node_as_rank  
0  28.532900  77.259600              16.0          17.0  
1  28.531895  77.259197   

In [2]:
df = bus.dropna(subset=['stop_lat', 'stop_lon']).copy()
print('With coords:', len(df), '/', len(bus))

# Time like 05:24:26:000 -> seconds from midnight
def time_to_sec(x):
    try:
        parts = str(x).strip().split(':')
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
        return h*3600 + m*60 + s
    except Exception:
        return np.nan

df['t_sec'] = df['time_raw'].map(time_to_sec)
df = df.dropna(subset=['t_sec'])
df = df.sort_values(['route_code', 'bus_number', 'day', 'month', 't_sec', 'node'])
print(df[['route_code','bus_number','node','t_sec','stop_lat','stop_lon']].head())

With coords: 419453 / 924375
     route_code  bus_number  node  t_sec   stop_lat   stop_lon
515         425  DL1PC 5373     1  29540  28.592517  77.236500
516         425  DL1PC 5373     2  30630  28.586133  77.241067
517         425  DL1PC 5373     3  30780  28.583417  77.240467
518         425  DL1PC 5373     4  30980  28.573217  77.237783
519         425  DL1PC 5373     5  31130  28.569100  77.235400


In [3]:
gcols = ['route_code', 'bus_number', 'month', 'day', 'direction']
df['dt'] = df.groupby(gcols)['t_sec'].diff()
df['ddist'] = df.groupby(gcols)['dist_to_junction_m'].diff()

# speed m/s from change in distance-to-junction / time (approx)
df['speed_mps'] = np.where(df['dt'] > 0, -df['ddist'] / df['dt'], np.nan)  # negative ddist => approaching
df.loc[df['speed_mps'] < 0, 'speed_mps'] = np.nan  # ignore moving away samples for approach speed
df['speed_mps'] = df['speed_mps'].clip(0, 30)

df['eta_sec'] = np.where(
    df['speed_mps'] > 0.5,
    df['dist_to_junction_m'] / df['speed_mps'],
    np.nan
)

# simple schedule proxy: compare to rolling median time at same node
df['node_median_t'] = df.groupby(['route_code', 'node'])['t_sec'].transform('median')
df['lateness_sec'] = df['t_sec'] - df['node_median_t']

print(df[['dist_to_junction_m','speed_mps','eta_sec','lateness_sec']].describe())

KeyError: 'Column not found: dist_to_junction_m'

In [4]:
r425 = df[df['route_code'] == 425]
junction_lat = r425['stop_lat'].median()
junction_lon = r425['stop_lon'].median()
print('Demo junction:', junction_lat, junction_lon)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

df['dist_to_junction_m'] = haversine_m(
    df['stop_lat'].values, df['stop_lon'].values,
    junction_lat, junction_lon
)
print(df['dist_to_junction_m'].describe())

Demo junction: 28.5411 77.2492
count    419453.000000
mean       3519.827929
std        1801.700959
min           4.884096
25%        2092.761489
50%        3285.836848
75%        4859.317645
max        6795.436398
Name: dist_to_junction_m, dtype: float64


In [5]:
gcols = ['route_code', 'bus_number', 'month', 'day', 'direction']
df['dt'] = df.groupby(gcols)['t_sec'].diff()
df['ddist'] = df.groupby(gcols)['dist_to_junction_m'].diff()

# speed m/s from change in distance-to-junction / time (approx)
df['speed_mps'] = np.where(df['dt'] > 0, -df['ddist'] / df['dt'], np.nan)  # negative ddist => approaching
df.loc[df['speed_mps'] < 0, 'speed_mps'] = np.nan  # ignore moving away samples for approach speed
df['speed_mps'] = df['speed_mps'].clip(0, 30)

df['eta_sec'] = np.where(
    df['speed_mps'] > 0.5,
    df['dist_to_junction_m'] / df['speed_mps'],
    np.nan
)

# simple schedule proxy: compare to rolling median time at same node
df['node_median_t'] = df.groupby(['route_code', 'node'])['t_sec'].transform('median')
df['lateness_sec'] = df['t_sec'] - df['node_median_t']

print(df[['dist_to_junction_m','speed_mps','eta_sec','lateness_sec']].describe())

       dist_to_junction_m      speed_mps        eta_sec   lateness_sec
count       419453.000000  190240.000000  161403.000000  419453.000000
mean          3519.827929       6.286623    1004.292438     895.737027
std           1801.700959       6.706673    1315.655026   15761.529421
min              4.884096      -0.000000       0.162803  -28150.000000
25%           2092.761489       1.201663     124.682544  -12920.000000
50%           3285.836848       3.885177     477.295319       0.000000
75%           4859.317645       9.597923    1364.584737   13768.000000
max           6795.436398      30.000000   10136.915192   34233.000000


In [6]:
def priority_label(row):
    # approaching corridor
    if not (50 <= row['dist_to_junction_m'] <= 800):
        return 'NONE'
    if pd.isna(row['eta_sec']) or row['eta_sec'] > 120:
        return 'NONE'
    # late bus near junction => ask priority
    if row['lateness_sec'] >= 60 and row['eta_sec'] <= 40:
        return 'EXTEND_GREEN'
    if row['lateness_sec'] >= 120 and row['eta_sec'] <= 70:
        return 'TRUNCATE_RED'
    if row['eta_sec'] <= 25 and row['lateness_sec'] >= 30:
        return 'EXTEND_GREEN'
    return 'NONE'

df['priority_action'] = df.apply(priority_label, axis=1)
print(df['priority_action'].value_counts())

priority_action
NONE            415421
TRUNCATE_RED      3044
EXTEND_GREEN       988
Name: count, dtype: int64


In [7]:
feat_cols = [
    'route_code', 'direction', 'node',
    'dist_to_junction_m', 'speed_mps', 'eta_sec', 'lateness_sec',
    't_sec'
]
ml = df.dropna(subset=['dist_to_junction_m', 'lateness_sec']).copy()
ml = ml[feat_cols + ['priority_action', 'bus_number', 'month', 'day']].copy()

# encode label
label_map = {'NONE': 0, 'EXTEND_GREEN': 1, 'TRUNCATE_RED': 2}
ml['y'] = ml['priority_action'].map(label_map)

# simple numeric features matrix
ml['direction_up'] = (ml['direction'].str.lower() == 'up').astype(int)

print('ML rows:', len(ml))
print(ml['priority_action'].value_counts())

ml.to_parquet(OUT / 'module1_ml_table.parquet', index=False)
ml.to_csv(OUT / 'module1_ml_table_sample.csv', index=False)

meta = {
    'rows': int(len(ml)),
    'label_counts': ml['priority_action'].value_counts().to_dict(),
    'junction_lat': float(junction_lat),
    'junction_lon': float(junction_lon),
    'features': ['dist_to_junction_m','speed_mps','eta_sec','lateness_sec','t_sec','direction_up','node','route_code']
}
(OUT / 'module1_ml_meta.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
print('STEP 4 OK' if len(ml) > 1000 and ml['y'].nunique() >= 2 else 'STEP 4 NEED LABEL FIX')

ML rows: 419453
priority_action
NONE            415421
TRUNCATE_RED      3044
EXTEND_GREEN       988
Name: count, dtype: int64
{
  "rows": 419453,
  "label_counts": {
    "NONE": 415421,
    "TRUNCATE_RED": 3044,
    "EXTEND_GREEN": 988
  },
  "junction_lat": 28.5411,
  "junction_lon": 77.2492,
  "features": [
    "dist_to_junction_m",
    "speed_mps",
    "eta_sec",
    "lateness_sec",
    "t_sec",
    "direction_up",
    "node",
    "route_code"
  ]
}
STEP 4 OK
